# 第17次课：GeoPandas 空间分析

**地学编程基础** | 2学时（90分钟）

---

## 📊 模块四进度

| 课次 | 主题 | 状态 |
|---|---|---|
| 第 16 次 | GeoPandas 基础（读、看、画、属性）| ✅ 已完成 |
| **第 17 次（今天）** | **GeoPandas 空间分析** | **📍 现在** |
| 第 18 次 | Rasterio 栅格遥感 | 待来 |
| 第 19 次 | 可视化与综合应用 | 待来 |

**今天的主题** —— 真正的 **GIS 思维**：不是按属性筛选，而是**按空间位置筛选**！

---

## 📚 本节课学习目标

完成本节课后，你应该能够：

1. **理解** CRS 转换的必要性（解决第 16 课的'平方度'问题）；
2. **掌握** `gdf.to_crs()` 转换坐标参考系；
3. **掌握**缓冲区分析 `gdf.geometry.buffer(distance)`；
4. **掌握**三种基础空间关系：`contains` / `within` / `intersects`；
5. **掌握**空间筛选 `gdf[gdf.geometry.within(area)]`；
6. **掌握**空间连接 `gpd.sjoin(left, right, how, predicate)`；
7. **掌握** `dissolve` 按属性合并几何（groupby 的空间版）；
8. **了解**几何运算（intersection / union / difference）；
9. **能够**完成完整的'空间分析工作流'。

---

## 📂 配套数据文件

本节课的目录结构：

```
lesson17/
├── Lesson17_GeoPandas空间分析.ipynb     ← 你正在看的
└── data/
    ├── provinces_simple.shp            ← 沿用第 16 课（7 个省份）
    ├── weather_stations.shp            ← 沿用第 16 课（7 个观测站）
    ├── weather_stations.csv            ← 沿用第 16 课
    └── rivers_simple.shp               ← ✨ 新增：4 条简化河流
```

---

## 🚀 震撼时刻 —— 解决'平方度'问题

**还记得第 16 课遗留的问题吗？**

```python
provinces['geo_area'] = provinces.geometry.area
# 北京的 geo_area = 3.20 ← 这是平方度（毫无意义）
# 真实的北京面积 = 16410 平方公里
```

**今天用一行 CRS 转换 解决它**：

In [ ]:
import geopandas as gpd
import warnings
warnings.filterwarnings('ignore')

provinces = gpd.read_file("data/provinces_simple.shp")

# 转换到 EPSG:3857（Web Mercator，单位米）
provinces_m = provinces.to_crs("EPSG:3857")

# 现在算面积 —— 单位是平方米
provinces_m["area_km2_calc"] = provinces_m.geometry.area / 1e6

provinces_m[["name", "area_km2", "area_km2_calc"]]

**🌟 看到了吗？**

- `area_km2`（属性表里写的）和 `area_km2_calc`（CRS 转换后算出来的）**接近一致**！
- 之前的'平方度'问题 → 一行 `to_crs()` 解决

**这就是 CRS 转换的力量** —— 让几何计算**有真实物理意义**。

**今天会学到**：
- CRS 转换
- 缓冲区分析（'某点周围 50km 范围'）
- 空间筛选（按位置筛选）
- 空间连接（点找面、面找点）
- dissolve（按属性合并几何）

---

## 一、CRS 转换 —— 让几何计算有意义

### 1.1 为什么需要转 CRS？

**EPSG:4326（经纬度）的局限**：
- 单位是**度** —— 算面积得平方度（无意义）
- 算长度得'弧度差'（无意义）
- '50km 缓冲区'根本没法表达

**投影坐标系（米单位）的优势**：
- 算面积 = 平方米 → /1e6 = 平方公里 ✅
- 算距离 = 米 ✅
- buffer(50000) = 50km 缓冲区 ✅

### 1.2 常用的投影坐标系

| EPSG | 名称 | 适用 | 单位 |
|---|---|---|---|
| 4326 | WGS84 经纬度 | 全球共享数据 | 度 |
| **3857** | Web Mercator | **全球都能用，单位米** | **米**（推荐！）|
| 32649 | UTM Zone 49N | 中国东部 | 米 |
| 32650 | UTM Zone 50N | 中国中部 | 米 |

**📌 经验法则**：
- 不知道选哪个？→ **先用 EPSG:3857**（全球可用）
- 局部精度要求高？→ 用对应的 UTM Zone

### 1.3 to_crs() —— 一行转换

In [ ]:
# 看原始 CRS
print("原始 CRS:", provinces.crs)

In [ ]:
# 转换到 EPSG:3857（Web Mercator）
provinces_3857 = provinces.to_crs("EPSG:3857")
print("转换后 CRS:", provinces_3857.crs)

# 看几何对象的坐标 —— 现在是米了！
print("\n转换前几何（度）：")
print(provinces.geometry.iloc[0])
print("\n转换后几何（米）：")
print(provinces_3857.geometry.iloc[0])

**🌟 关键观察**：
- 转换前坐标：`(115.5, 39.4)` 经纬度
- 转换后坐标：`(12858000, 4787000)` 米 ← 完全不同的数字！
- **同一个地方、不同的坐标系、不同的数字** —— 这就是 CRS 的本质

### 1.4 ⭐ '黄金工作流' —— 转 CRS 算几何

In [ ]:
# 黄金工作流：先转 CRS，再算几何
provinces_m = provinces.to_crs("EPSG:3857")

# 现在 area 单位是平方米
provinces_m["area_m2"] = provinces_m.geometry.area

# 转换为平方公里
provinces_m["area_km2_calc"] = provinces_m["area_m2"] / 1_000_000

# 对比属性表里的 area_km2 和算出来的
provinces_m[["name", "area_km2", "area_km2_calc"]]

**🌟 黄金工作流（请背下来）**：

```python
# 1. 读取（任何 CRS 都行）
gdf = gpd.read_file('xxx.shp')

# 2. 转到投影坐标系（米单位）
gdf_m = gdf.to_crs('EPSG:3857')

# 3. 做几何计算（area / length / buffer 等）
gdf_m['area_km2'] = gdf_m.geometry.area / 1e6
gdf_m['length_km'] = gdf_m.geometry.length / 1000
```

**📌 凡是涉及'真实距离/面积'的几何计算 —— 必须先 to_crs**！

### 1.5 算距离也要先转 CRS

In [ ]:
import geopandas as gpd
stations = gpd.read_file("data/weather_stations.shp")

# 想算北京到上海的真实距离
beijing = stations[stations["city"] == "北京"].geometry.iloc[0]
shanghai = stations[stations["city"] == "上海"].geometry.iloc[0]

# ❌ 错的方法：直接在 EPSG:4326 算距离
wrong_distance = beijing.distance(shanghai)
print(f"❌ EPSG:4326 距离: {wrong_distance:.2f}（这是度差，不是公里！）")

# ✅ 正确方法：先转 CRS
stations_m = stations.to_crs("EPSG:3857")
beijing_m = stations_m[stations_m["city"] == "北京"].geometry.iloc[0]
shanghai_m = stations_m[stations_m["city"] == "上海"].geometry.iloc[0]

right_distance_km = beijing_m.distance(shanghai_m) / 1000
print(f"✅ EPSG:3857 距离: {right_distance_km:.0f} km")

**📌 注意**：EPSG:3857（Web Mercator）在中纬度地区（中国）距离会**有几个百分点的误差**——但对教学和大多数场景已足够。**精度要求高的科研用 UTM**。

---

## 🎯 即时练习①

**练习目标**：熟悉 CRS 转换的标准工作流。

**练习题**：

1. 读取 `data/weather_stations.shp`；
2. 转换到 EPSG:3857；
3. 计算每个站点到北京的距离（公里），存为新列 `dist_to_bj_km`；
4. 找出距离北京最远的站点；
5. 读取 `data/rivers_simple.shp`，转 CRS 后计算每条河流的长度（公里）。

In [ ]:
# ===== 即时练习①.1-2 =====
import geopandas as gpd




In [ ]:
# ===== 即时练习①.3-4：计算各站到北京的距离 =====




In [ ]:
# ===== 即时练习①.5：每条河流的长度 =====




---

## 二、缓冲区分析 —— buffer

**'某点周围 50km 范围'是 GIS 最常见的需求**——这就是缓冲区分析。

### 2.1 buffer 的基本用法

In [ ]:
# 给观测站做 50km 缓冲区
stations_m = stations.to_crs("EPSG:3857")

# buffer(distance) —— distance 单位是 CRS 的单位（米）
buffer_50km = stations_m.geometry.buffer(50_000)    # 50000 米 = 50km

print("原几何类型:", type(stations_m.geometry.iloc[0]).__name__)
print("buffer 后类型:", type(buffer_50km.iloc[0]).__name__)

**🌟 关键观察**：
- `buffer(50_000)` —— 50km（注意单位！）
- **Point + buffer** → 圆形 Polygon（中心是点，半径 50km）
- **Polygon + buffer** → '扩张'后的 Polygon
- **LineString + buffer** → 沿线的'带状'Polygon

### 2.2 buffer + 画图 —— 真实例子

In [ ]:
import matplotlib.pyplot as plt

# 把缓冲区作为新 GeoDataFrame
stations_m_buffered = stations_m.copy()
stations_m_buffered["geometry"] = stations_m_buffered.geometry.buffer(50_000)

# 画图：省份 + 站点 + 50km 缓冲区
provinces_m = provinces.to_crs("EPSG:3857")

fig, ax = plt.subplots(figsize=(10, 8))

# 第 1 层：省份背景
provinces_m.plot(ax=ax, color="lightyellow", edgecolor="gray")

# 第 2 层：缓冲区（半透明）
stations_m_buffered.plot(ax=ax, color="red", alpha=0.3, edgecolor="red")

# 第 3 层：站点本身
stations_m.plot(ax=ax, color="black", markersize=30)

ax.set_title("观测站 50km 缓冲区", fontsize=14)
plt.show()

**🌟 缓冲区的实际用途**：
- '车站周围 1km 是步行影响区'
- '河流两岸 200m 是保护带'
- '机场周围 5km 是禁飞区'
- '观测站 50km 是有效观测范围'

### 2.3 ⭐ 河流的 20km 保护带

In [ ]:
rivers = gpd.read_file("data/rivers_simple.shp")
rivers_m = rivers.to_crs("EPSG:3857")

# 河流两岸 20km 缓冲区 —— 沿线的带状区域
rivers_buffered = rivers_m.copy()
rivers_buffered["geometry"] = rivers_buffered.geometry.buffer(20_000)

fig, ax = plt.subplots(figsize=(10, 8))
provinces_m.plot(ax=ax, color="lightyellow", edgecolor="gray")
rivers_buffered.plot(ax=ax, color="lightblue", alpha=0.5, edgecolor="blue")
rivers_m.plot(ax=ax, color="darkblue", linewidth=2)

ax.set_title("河流 20km 保护带", fontsize=14)
plt.show()

---

## 三、空间关系 —— contains / within / intersects

**真正的 GIS 思维 —— 按空间关系筛选**！

### 3.1 三种最核心的空间关系

| 关系 | 含义 | 例子 |
|---|---|---|
| **A.contains(B)** | A 包含 B | 省份是否包含某个观测站？|
| **A.within(B)** | A 在 B 内 | 观测站是否在某个省份内？|
| **A.intersects(B)** | A 和 B 有交集 | 河流是否经过某个省份？|

**📌 重要**：`A.contains(B)` 和 `B.within(A)` 是**等价**的！只是视角不同。

（还有 8 种空间关系：disjoint/touches/crosses/overlaps/equals/covers/coveredby/contains_properly，今天先掌握 3 种核心）

### 3.2 contains 例子：省份包含哪些站点？

In [ ]:
# 取广东省的几何
guangdong_poly = provinces[provinces["name"] == "广东"].geometry.iloc[0]

# 检查每个站点是否在广东省内
stations["in_guangdong"] = stations.geometry.within(guangdong_poly)
stations[["city", "region", "in_guangdong"]]

**🌟 看到了吗？**——广州显示 `True`，其他都是 `False`——**这就是空间筛选的基础**。

### 3.3 within 例子：站点在某区域内

In [ ]:
# 把华南区域（广东+海南）合并成一个几何
from shapely.ops import unary_union

huanan = provinces[provinces["region"] == "华南"]
huanan_geom = unary_union(huanan.geometry)    # 合并多个几何为一个

# 哪些站点在华南区域内？
in_huanan = stations[stations.geometry.within(huanan_geom)]
print(f"华南区域内的站点（{len(in_huanan)} 个）：")
print(in_huanan[["city", "region"]])

### 3.4 intersects 例子：河流经过哪些省份？

In [ ]:
# 取长江几何
yangtze = rivers[rivers["name"].str.contains("长江")].geometry.iloc[0]

# 哪些省份与长江有交集？
provinces["intersects_yangtze"] = provinces.geometry.intersects(yangtze)
yangtze_provinces = provinces[provinces["intersects_yangtze"]]

print("长江-华东段经过的省份：")
print(yangtze_provinces[["name", "region"]])

**🌟 这就是真正的 GIS 分析**——'河流经过哪些省？'是一个**空间问题**，不能靠属性筛选解决。

### 3.5 ⭐⭐⭐ 空间筛选 vs 属性筛选

**今天最重要的认知升级**：

| | 属性筛选 | 空间筛选 |
|---|---|---|
| 依据 | 列的值 | 几何位置 |
| 例子 | `df[df['region'] == '华南']` | `df[df.within(area)]` |
| 限制 | 只能用现有属性 | **不需要预先标注**|
| 何时用 | 数据已分类 | **想根据位置发现新关系** |

**📌 记住**：
- 已知'广州属于华南'（属性已有）→ **属性筛选**
- 想知道'广州站点周围 50km 内有哪些其他站点'（属性表里没有）→ **空间筛选**

---

## 🎯 即时练习②

**练习目标**：综合运用空间关系。

**练习题**：

1. 用 `unary_union` 把'华南'+'华东'两个区域合并成一个几何叫 `south_geom`；
2. 找出在 `south_geom` 内的所有观测站（用 within）；
3. 把河流转 CRS 后做 30km 缓冲区，存成新 GeoDataFrame `rivers_buffer`；
4. 哪些观测站在某条河流的 30km 缓冲区内？（提示：用 unary_union 合并所有缓冲区，再 within）

In [ ]:
# ===== 即时练习②.1-2：华南华东合并并筛选 =====
import geopandas as gpd
from shapely.ops import unary_union

provinces = gpd.read_file("data/provinces_simple.shp")
stations = gpd.read_file("data/weather_stations.shp")




In [ ]:
# ===== 即时练习②.3：河流 30km 缓冲区 =====




In [ ]:
# ===== 即时练习②.4：找出在河流缓冲区内的站点 =====




---

## 四、空间连接 —— sjoin（最强大的空间分析工具）

**SQL 里你用 JOIN 连接两张表**——**GIS 里用 sjoin 按空间关系连接两个 GeoDataFrame**！

### 4.1 基本场景：给点加上所在面的属性

In [ ]:
# 场景：给每个观测站标注'它属于哪个省'
# 这是 GIS 工程师每天都要做的事

# 一行 sjoin 解决
joined = gpd.sjoin(
    stations,                                  # 左表（点）
    provinces[["name", "region", "geometry"]],  # 右表（面）—— 只取需要的列
    how="left",                                # 连接方式
    predicate="within"                         # 空间关系
)

joined[["city", "name", "region"]]

**🌟 看到了吗？**——每个站点都被标注了它所在的省份名称！

**这就是 sjoin 的力量**——**给点添加面的属性**——是 GIS 工程师最常用的操作。

### 4.2 sjoin 的 4 个关键参数

```python
gpd.sjoin(left, right, how, predicate)
```

**参数详解**：

| 参数 | 含义 | 常用值 |
|---|---|---|
| `left` | 主表（结果保留它的几何）| 你的核心数据 |
| `right` | 副表（提供要附加的属性）| 参考数据 |
| `how` | 连接方式 | `'left'`（保留所有左表）/ `'inner'`（只要匹配的）|
| `predicate` | 空间关系 | `'within'`（点在面内）/ `'intersects'`（相交）/ `'contains'`（包含）|

**📌 经验法则**：
- '点找面'（给站点加省份属性）→ `predicate='within'`
- '面找面'（找有交集的）→ `predicate='intersects'`
- '面找点'（找面内的所有点）→ 把面作为左表，`predicate='contains'`

### 4.3 ⭐ how='left' vs how='inner'

In [ ]:
# how='left' —— 保留所有左表的行（不匹配的右表列为 NaN）
left_join = gpd.sjoin(stations, provinces[["name", "geometry"]], how="left", predicate="within")
print(f"left join: {len(left_join)} 行（=stations 的行数）")

# how='inner' —— 只保留匹配上的行
inner_join = gpd.sjoin(stations, provinces[["name", "geometry"]], how="inner", predicate="within")
print(f"inner join: {len(inner_join)} 行（只保留有匹配的）")

**📌 经验法则**：
- 想知道'**哪些没匹配上**'→ `how='left'`（看 NaN 行）
- 只关心'**匹配上的**'→ `how='inner'`

### 4.4 实战：sjoin + groupby（GeoPandas 的两大神器组合）

In [ ]:
# 问题：每个区域内的观测站平均海拔是多少？
# 步骤：sjoin 把站点关联到省份 → groupby 按区域聚合

# 步骤 1：sjoin 给站点加上 region 属性
joined = gpd.sjoin(
    stations,
    provinces[["region", "geometry"]],
    how="inner",
    predicate="within"
)

# 步骤 2：按 region 聚合
result = joined.groupby("region").agg({
    "elevation": "mean",
    "rainfall": "mean",
    "city": "count"
}).rename(columns={"city": "station_count"})

result

**🌟 这就是 GIS 数据分析的'核心组合'**：

1. **sjoin** —— 把空间数据关联起来（点关联到面）
2. **groupby** —— 按属性聚合（第 15 课的技能）

**两个组合 = 真正的空间分析报告生成器**！

---

## 五、dissolve —— groupby 的空间版

**第 15 课你学了 groupby**——按属性聚合**数值列**。

**今天 dissolve**——按属性聚合**几何**！

**例子**：把 7 个省份按 region 合并成 4 个区域。

In [ ]:
# 按 region 合并
regions = provinces.dissolve(by="region")
regions

**🌟 关键观察**：
- 7 个省份 → 4 个区域
- **几何被'融合'**（华北的 2 个省合并成 1 个面）
- 默认保留每组**第一个**值作为属性

**📌 dissolve 是 GeoPandas 特有的'空间聚合'** —— 普通 Pandas 没有这个功能。

### 5.1 dissolve + aggfunc —— 同时聚合属性

In [ ]:
# 合并几何 + 聚合数值列（呼应第 15 课的 agg）
regions = provinces.dissolve(
    by="region",
    aggfunc={
        "area_km2": "sum",       # 区域总面积
        "pop_millio": "sum"      # 区域总人口
    }
)
regions

**🌟 这就是 dissolve 的强大**：
- 一行同时做了：**几何合并** + **属性聚合**
- 4 个区域：每个区域的面积总和、人口总和、合并后的几何

In [ ]:
# 画图对比
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 左：合并前
provinces.plot(ax=axes[0], column="region", cmap="Set2", edgecolor="black", legend=True)
axes[0].set_title("合并前（7 省份）", fontsize=14)

# 右：合并后
regions = regions.reset_index()    # 把 region 从 index 拿回成普通列
regions.plot(ax=axes[1], column="region", cmap="Set2", edgecolor="black", legend=True)
axes[1].set_title("合并后（4 区域）", fontsize=14)

plt.tight_layout()
plt.show()

**📌 dissolve 的实战用途**：
- 行政区合并（市 → 省 → 大区）
- 土地利用类型合并（多个地块 → 一个大类）
- 流域合并（小流域 → 干流流域）

---

## 六、几何运算（点到为止）

**真正的 GIS 计算 —— 几何对象的'集合运算'**。

| 运算 | 含义 | 例子 |
|---|---|---|
| `intersection` | 交集 | A ∩ B |
| `union` | 并集 | A ∪ B |
| `difference` | 差集 | A − B |

**今天先认识这些**——具体用法可以以后碰到再学。

In [ ]:
from shapely.geometry import Polygon

# 两个简单的方框
A = Polygon([(0, 0), (4, 0), (4, 4), (0, 4)])
B = Polygon([(2, 2), (6, 2), (6, 6), (2, 6)])

print("A:", A)
print("B:", B)

# 三种运算
print("\nA ∩ B（交集）:", A.intersection(B))
print("A ∪ B（并集）:", A.union(B))
print("A − B（差集）:", A.difference(B))

---

## 🎯 即时练习③（综合实战）

**练习目标**：完成真实的'空间分析工作流'。

**情境**：你被要求做一份'**沿海湿润地区分析报告**'。

**任务步骤**：

**步骤1**：读取 `provinces_simple.shp` 和 `weather_stations.shp`；

**步骤2**：把 `provinces` 按 `region` dissolve（合并）—— 注意 aggfunc 把 area_km2 和 pop_millio 求和；

**步骤3**：用 `sjoin` 给每个站点关联所属的 region（`predicate='within'`，`how='inner'`）；

**步骤4**：筛选出 `rainfall > 1000` 的'湿润站点'（属性筛选）；

**步骤5**：再筛选出在 '华东' 或 '华南' 区域的湿润站点（用 isin）；

**步骤6**：把结果转 CRS 到 EPSG:3857，做 30km 缓冲区；

**步骤7**：画一张专题地图：4 大区域 + 湿润站点 + 30km 缓冲区；

**步骤8**：导出湿润站点为 `output_humid_coastal_stations.gpkg`。

In [ ]:
# ===== 即时练习③ 综合实战 =====
import geopandas as gpd
import matplotlib.pyplot as plt

# 步骤1：读取




In [ ]:
# 步骤2：dissolve 合并省份成 4 区域




In [ ]:
# 步骤3：sjoin 给站点加 region




In [ ]:
# 步骤4-5：筛选 humid + 华东华南




In [ ]:
# 步骤6：转 CRS + 30km 缓冲区




In [ ]:
# 步骤7：画图




In [ ]:
# 步骤8：导出 GeoPackage




---

## 七、本节课小结

### 知识点回顾

| 知识点 | 关键语法 |
|---|---|
| CRS 转换 | `gdf.to_crs('EPSG:3857')` |
| 黄金工作流 | 转 CRS → 算几何 → 单位换算 |
| 缓冲区 | `gdf.geometry.buffer(distance)` |
| 包含关系 | `A.contains(B)` / `B.within(A)` |
| 相交关系 | `A.intersects(B)` |
| 空间筛选 | `gdf[gdf.within(area)]` |
| 空间连接 | `gpd.sjoin(left, right, how, predicate)` |
| 几何合并 | `gdf.dissolve(by='col', aggfunc=...)` |
| 几何运算 | `intersection / union / difference` |

### 重点强调（重要程度从高到低）

1. ⭐⭐⭐ **黄金工作流：先 to_crs，再几何计算** —— 解决平方度问题
2. ⭐⭐⭐ **空间筛选 vs 属性筛选** —— GIS 思维的核心
3. ⭐⭐⭐ **sjoin = 给点找面** —— GIS 工程师每天用
4. ⭐⭐ **buffer 单位是 CRS 单位** —— 必须先转米单位
5. ⭐⭐ **dissolve = groupby 的空间版** —— 第 15 课的扩展
6. ⭐⭐ **3 种核心空间关系** —— contains/within/intersects
7. ⭐ **sjoin 的 how 和 predicate** —— left vs inner、within vs intersects

### 课后作业

**1. 【基础】CRS 转换与几何计算（必做）**

用 `data/provinces_simple.shp`：
1. 转换到 EPSG:3857；
2. 计算每个省份的真实面积（平方公里），存为 `area_km2_calc`；
3. 对比属性表里的 `area_km2`，计算误差百分比；
4. 计算每个省份的周长（公里）。

**2. 【进阶】空间分析（必做）**

1. 用 `data/weather_stations.shp`，先转到 EPSG:3857；
2. 给每个站点做 100km 缓冲区；
3. 用 sjoin 找出'每个站点的 100km 缓冲区内还有哪些其他站点'；
4. 用 dissolve 把所有缓冲区合并成一个几何。

**3. 【挑战】河流影响分析（必做）**

1. 读取 `data/rivers_simple.shp` 和 `data/provinces_simple.shp`；
2. 用 sjoin（`predicate='intersects'`）找出每条河流经过的所有省份；
3. 转 CRS 后给河流做 50km 缓冲区；
4. 用 sjoin 找出在'任意河流 50km 缓冲区内'的所有观测站；
5. 画一张专题地图：省份淡灰色 + 河流蓝色 + 50km 缓冲区半透明 + 受影响的站点红色；
6. 导出受影响站点为 `output_riverside_stations.gpkg`。

---

下节课预告：**第18次课 Rasterio 栅格遥感数据** —— 真正的卫星影像处理！读 GeoTIFF、计算 NDVI、做栅格统计 —— 与今天的矢量分析互补。

---
---

# 📎 附录：参考答案

> **建议先独立完成所有练习题再查看答案。**

In [ ]:
# ===== 即时练习①.1-2 =====
import geopandas as gpd

stations = gpd.read_file("data/weather_stations.shp")
stations_m = stations.to_crs("EPSG:3857")
print("原始 CRS:", stations.crs)
print("转换后 CRS:", stations_m.crs)

In [ ]:
# ===== 即时练习①.3-4 =====
# 取北京站点几何
beijing_m = stations_m[stations_m["city"] == "北京"].geometry.iloc[0]

# 计算每个站到北京的距离
stations_m["dist_to_bj_km"] = stations_m.geometry.distance(beijing_m) / 1000

# 排序
result = stations_m[["city", "dist_to_bj_km"]].sort_values("dist_to_bj_km", ascending=False)
print(result)
print(f"\n距离北京最远的站点：{result.iloc[0]['city']}（{result.iloc[0]['dist_to_bj_km']:.0f} km）")

In [ ]:
# ===== 即时练习①.5 =====
rivers = gpd.read_file("data/rivers_simple.shp")
rivers_m = rivers.to_crs("EPSG:3857")
rivers_m["length_km"] = rivers_m.geometry.length / 1000
print(rivers_m[["name", "length_km"]])

In [ ]:
# ===== 即时练习②.1-2 =====
import geopandas as gpd
from shapely.ops import unary_union

provinces = gpd.read_file("data/provinces_simple.shp")
stations = gpd.read_file("data/weather_stations.shp")

# 1. 合并华南+华东
south = provinces[provinces["region"].isin(["华南", "华东"])]
south_geom = unary_union(south.geometry)

# 2. 找出在 south_geom 内的站点
in_south = stations[stations.geometry.within(south_geom)]
print(f"在华东+华南内的站点（{len(in_south)} 个）：")
print(in_south[["city", "region"]])

In [ ]:
# ===== 即时练习②.3 =====
rivers = gpd.read_file("data/rivers_simple.shp")
rivers_m = rivers.to_crs("EPSG:3857")

rivers_buffer = rivers_m.copy()
rivers_buffer["geometry"] = rivers_buffer.geometry.buffer(30_000)    # 30km
print(rivers_buffer[["name", "geometry"]].head())

In [ ]:
# ===== 即时练习②.4 =====
# 把所有缓冲区合并
all_buffer = unary_union(rivers_buffer.geometry)

# 把站点也转到同 CRS
stations_m = stations.to_crs("EPSG:3857")

# 找出在缓冲区内的站点
near_river = stations_m[stations_m.geometry.within(all_buffer)]
print(f"靠近河流的站点（{len(near_river)} 个）：")
print(near_river[["city", "region"]])

In [ ]:
# ===== 即时练习③ 综合实战 参考答案 =====
import geopandas as gpd
import matplotlib.pyplot as plt

# 步骤1：读取
provinces = gpd.read_file("data/provinces_simple.shp")
stations = gpd.read_file("data/weather_stations.shp")

# 步骤2：dissolve 合并
regions = provinces.dissolve(
    by="region",
    aggfunc={"area_km2": "sum", "pop_millio": "sum"}
).reset_index()
print("=== 4 区域合并结果 ===")
print(regions[["region", "area_km2", "pop_millio"]])

In [ ]:
# 步骤3：sjoin 给站点加 region
stations_with_region = gpd.sjoin(
    stations,
    regions[["region", "geometry"]],
    how="inner",
    predicate="within"
)
print("=== sjoin 结果 ===")
print(stations_with_region[["city", "region"]])

In [ ]:
# 步骤4-5：筛选湿润 + 华东华南
humid = stations_with_region[stations_with_region["rainfall"] > 1000]
humid_coastal = humid[humid["region"].isin(["华东", "华南"])]
print(f"=== 沿海湿润站点（{len(humid_coastal)} 个）===")
print(humid_coastal[["city", "region", "rainfall"]])

In [ ]:
# 步骤6：转 CRS + 30km 缓冲区
humid_m = humid_coastal.to_crs("EPSG:3857")
humid_buffer = humid_m.copy()
humid_buffer["geometry"] = humid_buffer.geometry.buffer(30_000)

In [ ]:
# 步骤7：画图
regions_m = regions.to_crs("EPSG:3857")

fig, ax = plt.subplots(figsize=(12, 9))
regions_m.plot(ax=ax, column="region", cmap="Set3", edgecolor="black", legend=True, alpha=0.5)
humid_buffer.plot(ax=ax, color="blue", alpha=0.3, edgecolor="darkblue")
humid_m.plot(ax=ax, color="red", markersize=120, edgecolor="black")
ax.set_title("沿海湿润地区分析（华东+华南，年降水>1000mm）", fontsize=14)
plt.show()

In [ ]:
# 步骤8：导出 GeoPackage
humid_coastal.to_file("output_humid_coastal_stations.gpkg", driver="GPKG", layer="stations")
print("output_humid_coastal_stations.gpkg 已写入！")
print(humid_coastal[["city", "region", "rainfall"]])

---

## 课后作业参考答案

In [ ]:
# ===== 课后作业 1 参考答案 =====
import geopandas as gpd

provinces = gpd.read_file("data/provinces_simple.shp")
provinces_m = provinces.to_crs("EPSG:3857")

# 1-2. 计算真实面积
provinces_m["area_km2_calc"] = provinces_m.geometry.area / 1e6

# 3. 误差百分比
provinces_m["error_pct"] = (
    (provinces_m["area_km2_calc"] - provinces_m["area_km2"]) / provinces_m["area_km2"] * 100
)

# 4. 周长（公里）
provinces_m["perimeter_km"] = provinces_m.geometry.length / 1000

print(provinces_m[["name", "area_km2", "area_km2_calc", "error_pct", "perimeter_km"]].round(1))

In [ ]:
# ===== 课后作业 2 参考答案 =====
import geopandas as gpd

stations = gpd.read_file("data/weather_stations.shp")
stations_m = stations.to_crs("EPSG:3857")

# 2. 100km 缓冲区
buffer_gdf = stations_m.copy()
buffer_gdf["geometry"] = buffer_gdf.geometry.buffer(100_000)

# 3. 找每个站点 100km 内的其他站点
neighbors = gpd.sjoin(
    stations_m,
    buffer_gdf[["city", "geometry"]].rename(columns={"city": "buffer_owner"}),
    how="inner",
    predicate="within"
)
# 排除自己
neighbors = neighbors[neighbors["city"] != neighbors["buffer_owner"]]
print("=== 100km 内的相邻站点 ===")
print(neighbors[["buffer_owner", "city"]])

# 4. dissolve 合并所有缓冲区
buffer_gdf["all"] = "all"
merged = buffer_gdf.dissolve(by="all")
print(f"\n合并所有缓冲区后：{len(merged)} 个几何对象")

In [ ]:
# ===== 课后作业 3 参考答案 =====
import geopandas as gpd
import matplotlib.pyplot as plt

# 1. 读取
provinces = gpd.read_file("data/provinces_simple.shp")
rivers = gpd.read_file("data/rivers_simple.shp")
stations = gpd.read_file("data/weather_stations.shp")

# 2. sjoin 找河流经过的省份
river_provinces = gpd.sjoin(
    provinces[["name", "region", "geometry"]],
    rivers[["name", "geometry"]].rename(columns={"name": "river_name"}),
    how="inner",
    predicate="intersects"
)
print("=== 河流经过的省份 ===")
print(river_provinces[["name", "river_name"]])

In [ ]:
# 3. 转 CRS + 50km 缓冲区
rivers_m = rivers.to_crs("EPSG:3857")
rivers_buffer = rivers_m.copy()
rivers_buffer["geometry"] = rivers_buffer.geometry.buffer(50_000)

# 4. 找受影响的站点
stations_m = stations.to_crs("EPSG:3857")
affected = gpd.sjoin(
    stations_m,
    rivers_buffer[["name", "geometry"]].rename(columns={"name": "river"}),
    how="inner",
    predicate="within"
).drop_duplicates("city")
print(f"\n=== 受河流影响的站点（{len(affected)} 个）===")
print(affected[["city", "region", "river"]])

In [ ]:
# 5. 画专题地图
provinces_m = provinces.to_crs("EPSG:3857")
fig, ax = plt.subplots(figsize=(12, 9))

provinces_m.plot(ax=ax, color="lightgray", edgecolor="darkgray", linewidth=0.5)
rivers_buffer.plot(ax=ax, color="lightblue", alpha=0.5, edgecolor="blue")
rivers_m.plot(ax=ax, color="darkblue", linewidth=2)
affected.plot(ax=ax, color="red", markersize=120, edgecolor="black", marker="^")

ax.set_title("河流 50km 影响范围分析", fontsize=14)
plt.show()

# 6. 导出
affected.to_file("output_riverside_stations.gpkg", driver="GPKG", layer="stations")
print("\noutput_riverside_stations.gpkg 已写入！")